# ✊✋✌️ Schere-Stein-Papier – Kann ein Computer das Muster knacken?

**Worum geht's?** Wir bauen einen Computer-Spieler, der lernt, unsere Wahl vorherzusagen – nur durch Ausprobieren und Übung. Das nennt man **Reinforcement Learning** (Lernen durch Belohnung).

---

## Die Spielregeln (kennst du schon!)

Zwei Spieler wählen gleichzeitig: **Schere ✌️, Stein ✊ oder Papier ✋**

| | schlägt |
|---|---|
| ✌️ Schere | ✋ Papier |
| ✊ Stein | ✌️ Schere |
| ✋ Papier | ✊ Stein |

Bei gleicher Wahl: Unentschieden. Wir spielen **10 Runden** – wer mehr Runden gewinnt, gewinnt das Spiel.

---

## 🤔 Der Trick an diesem Notebook

Normalerweise ist Schere-Stein-Papier reines Glücksspiel – wenn beide zufällig wählen, kann niemand vorhersagen, was kommt.

**Aber:** Unser Computergegner spielt nicht ganz zufällig. Er hat eine kleine Angewohnheit, ein **Muster**. Menschen haben das oft auch, ohne es zu merken!

Unsere Aufgabe: Ein Computer-Spieler (ein **Agent**) soll dieses Muster selbst entdecken – nur indem er viele Runden spielt und sich merkt, was funktioniert hat.

---
## Teil 1: Das Spiel bauen

In [ ]:
# Diese Zelle zuerst ausführen - sie lädt alles, was wir brauchen
import random
import matplotlib.pyplot as plt
from collections import defaultdict

print("✅ Bereit zum Spielen!")

In [ ]:
# Die drei Wahlmöglichkeiten - wir benutzen Zahlen, das macht es dem Computer leichter
SCHERE = 0
STEIN = 1
PAPIER = 2

namen = {SCHERE: "Schere ✌️", STEIN: "Stein ✊", PAPIER: "Papier ✋"}


def wer_gewinnt(wahl_a, wahl_b):
    """
    Vergleicht zwei Wahlen.
    Gibt zurück: +1 wenn A gewinnt, -1 wenn B gewinnt, 0 bei Unentschieden.
    """
    if wahl_a == wahl_b:
        return 0
    # Schere schlägt Papier, Stein schlägt Schere, Papier schlägt Stein
    gewinnt_gegen = {SCHERE: PAPIER, STEIN: SCHERE, PAPIER: STEIN}
    if gewinnt_gegen[wahl_a] == wahl_b:
        return 1
    else:
        return -1


# Kurzer Test
print("Schere vs. Papier:", "A gewinnt" if wer_gewinnt(SCHERE, PAPIER) == 1 else "?")
print("Stein vs. Schere:  ", "A gewinnt" if wer_gewinnt(STEIN, SCHERE) == 1 else "?")
print("Papier vs. Papier: ", "Unentschieden" if wer_gewinnt(PAPIER, PAPIER) == 0 else "?")

---
## Teil 2: Der geheimnisvolle Gegner

Unser Computergegner spielt nach folgender Angewohnheit:

> **"Nach einem Sieg spiele ich mit hoher Wahrscheinlichkeit dasselbe nochmal."**

Das ist gar nicht so unrealistisch – viele Menschen machen das tatsächlich unbewusst! Wenn etwas funktioniert hat, wiederholt man es gerne.

Schauen wir uns das Verhalten an:

In [ ]:
class MusterGegner:
    """
    Ein Computergegner mit einer Angewohnheit:
    Nach einem Sieg wiederholt er mit 70% Wahrscheinlichkeit seine letzte Wahl.
    Sonst (und ganz am Anfang) wählt er zufällig.
    """

    def __init__(self, wiederhol_wahrscheinlichkeit=0.7):
        self.letzte_wahl = None
        self.hat_gewonnen = False
        self.wiederhol_wahrscheinlichkeit = wiederhol_wahrscheinlichkeit

    def waehle(self):
        if self.hat_gewonnen and random.random() < self.wiederhol_wahrscheinlichkeit:
            wahl = self.letzte_wahl  # Wiederholt die erfolgreiche Wahl
        else:
            wahl = random.choice([SCHERE, STEIN, PAPIER])  # Zufällig
        return wahl

    def merke_ergebnis(self, eigene_wahl, hat_gewonnen):
        self.letzte_wahl = eigene_wahl
        self.hat_gewonnen = hat_gewonnen


# Schauen wir uns 15 Spielzüge des Gegners an (gegen einen zufälligen Spieler)
print("Beispielverlauf des Gegners:\n")
gegner = MusterGegner()
for runde in range(15):
    wahl_gegner = gegner.waehle()
    wahl_zufall = random.choice([SCHERE, STEIN, PAPIER])
    ergebnis = wer_gewinnt(wahl_gegner, wahl_zufall)
    gegner_gewinnt = (ergebnis == 1)
    print(f"Runde {runde+1:2d}: Gegner spielt {namen[wahl_gegner]:12s} "
          f"{'(gewinnt → wiederholt evtl.)' if gegner_gewinnt else ''}")
    gegner.merke_ergebnis(wahl_gegner, gegner_gewinnt)

**Habt ihr das Muster erkannt?** Wenn der Gegner gewinnt, taucht seine Wahl in der nächsten Runde oft wieder auf!

Genau dieses Muster soll unser lernender Agent jetzt selbst entdecken – ohne dass wir es ihm verraten.

---
## Teil 3: Der lernende Agent

### Wie soll der Agent lernen?

Der Agent merkt sich eine einfache Sache: **"Was hat der Gegner in der letzten Runde gespielt?"** Das nennen wir den **Zustand**.

Für jeden Zustand führt der Agent eine Punktzahl für jede seiner drei möglichen Antworten. Diese Punktzahl nennen wir **Q-Wert** – ein Maß dafür, *wie gut diese Antwort bisher funktioniert hat*.

**Die Lernregel in einfachen Worten:**

- Hat eine Antwort zum Sieg geführt → Punktzahl für diese Antwort steigt ein bisschen
- Hat eine Antwort zur Niederlage geführt → Punktzahl sinkt ein bisschen
- Bei Unentschieden → kaum Veränderung

Mit der Zeit sammeln sich für gute Antworten höhere Punktzahlen an. Der Agent wählt dann meistens die Antwort mit der höchsten Punktzahl – aber probiert ab und zu auch etwas anderes aus, um nichts zu verpassen.

In [ ]:
class LernenderAgent:
    """
    Ein einfacher Q-Learning Agent für Schere-Stein-Papier.
    
    Merkt sich für jeden Zustand (= gegnerische letzte Wahl) eine Punktzahl
    pro eigener Antwortmöglichkeit.
    """

    def __init__(self, lerntempo=0.02, zufallsquote=1.0, zufallsquote_minimum=0.02, abkuehlung=0.9997):
        self.punkte = defaultdict(float)        # Punktzahl pro (Zustand, Antwort)
        self.lerntempo = lerntempo               # Wie stark wirkt jede neue Erfahrung?
        self.zufallsquote = zufallsquote         # Wie oft wird einfach ausprobiert?
        self.zufallsquote_minimum = zufallsquote_minimum
        self.abkuehlung = abkuehlung             # Zufallsquote sinkt mit der Zeit

    def waehle_antwort(self, zustand):
        """Wählt eine Antwort: meistens die bisher beste, manchmal zufällig zum Ausprobieren."""
        if random.random() < self.zufallsquote:
            return random.choice([SCHERE, STEIN, PAPIER])  # Ausprobieren
        # Bewährtes nutzen: wähle die Antwort mit der höchsten Punktzahl
        return max([SCHERE, STEIN, PAPIER], key=lambda a: self.punkte[(zustand, a)])

    def lerne(self, zustand, antwort, belohnung):
        """Passt die Punktzahl nach einer Runde an."""
        alte_punktzahl = self.punkte[(zustand, antwort)]
        self.punkte[(zustand, antwort)] = alte_punktzahl + self.lerntempo * (belohnung - alte_punktzahl)

        # Mit der Zeit weniger ausprobieren, mehr Bewährtes nutzen
        if self.zufallsquote > self.zufallsquote_minimum:
            self.zufallsquote *= self.abkuehlung


print("✅ Lernender Agent ist bereit")

---
## Teil 4: Training – viele, viele Runden üben

Jetzt lassen wir unseren Agenten gegen den Muster-Gegner antreten – nicht nur 10 Runden, sondern **50.000 Runden**! So viel Übung braucht ein Computer, um ein verlässliches Muster zu erkennen.

Das passiert in jeder Runde:

1. Agent merkt sich, was der Gegner zuletzt gespielt hat (= **Zustand**)
2. Agent wählt eine Antwort
3. Gegner wählt (nach seiner Angewohnheit)
4. Wer gewinnt, wird ausgewertet → **Belohnung**: +1 / -1 / 0
5. Agent passt seine Punktzahlen an

In [ ]:
def trainiere_agent(anzahl_runden=50000):
    """
    Lässt den Agenten gegen den Muster-Gegner trainieren.
    Gibt den trainierten Agenten und eine Lern-Statistik zurück.
    """
    agent = LernenderAgent()
    gegner = MusterGegner()

    # Statistik für die Lernkurve (alle 200 Runden ein Messpunkt)
    mess_intervall = 200
    siege_agent = siege_gegner = unentschieden = 0
    verlauf_siege = []
    verlauf_zufallsquote = []

    zustand = None  # Zu Beginn kennen wir die letzte Wahl des Gegners noch nicht

    for runde in range(1, anzahl_runden + 1):
        # Zustand: Was hat der Gegner zuletzt gespielt? (None = Spielanfang)
        antwort = agent.waehle_antwort(zustand)
        wahl_gegner = gegner.waehle()

        ergebnis = wer_gewinnt(antwort, wahl_gegner)
        belohnung = ergebnis  # +1, -1, oder 0 - direkt als Belohnung verwendbar

        agent.lerne(zustand, antwort, belohnung)
        gegner.merke_ergebnis(wahl_gegner, ergebnis == -1)  # Gegner gewinnt, wenn ergebnis == -1

        if ergebnis == 1:
            siege_agent += 1
        elif ergebnis == -1:
            siege_gegner += 1
        else:
            unentschieden += 1

        zustand = wahl_gegner  # Für die nächste Runde merken wir uns die gegnerische Wahl

        if runde % mess_intervall == 0:
            verlauf_siege.append(siege_agent / mess_intervall * 100)
            verlauf_zufallsquote.append(agent.zufallsquote)
            siege_agent = siege_gegner = unentschieden = 0

    statistik = {"siege": verlauf_siege, "zufallsquote": verlauf_zufallsquote, "intervall": mess_intervall}
    return agent, statistik


print("Training läuft ...")
agent, statistik = trainiere_agent(anzahl_runden=50000)
print("✅ Training abgeschlossen!")

---
## Teil 5: Hat der Agent das Muster geknackt?

Wir schauen uns an, wie sich die Siegquote des Agenten über die Zeit verändert hat. Zur Erinnerung: Bei einem rein zufälligen Gegner läge die Siegquote bei etwa 33% (genauso oft gewinnt, verliert oder unentschieden). Unser Gegner hat aber ein Muster – kann der Agent darüber liegen?

In [ ]:
def zeige_lernkurve(statistik):
    runden_achse = [i * statistik["intervall"] for i in range(1, len(statistik["siege"]) + 1)]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Plot 1: Siegquote über die Zeit
    ax = axes[0]
    ax.plot(runden_achse, statistik["siege"], color="#1565C0", linewidth=2)
    ax.axhline(y=33.3, color="gray", linestyle="--", alpha=0.7, label="Zufall (33%)")
    ax.set_xlabel("Trainingsrunde")
    ax.set_ylabel("Siegquote des Agenten (%)")
    ax.set_title("Wird der Agent besser?")
    ax.legend()
    ax.set_ylim(0, 80)
    ax.grid(alpha=0.3)

    # Plot 2: Wie oft probiert der Agent noch aus?
    ax = axes[1]
    werte = [z * 100 for z in statistik["zufallsquote"]]
    ax.plot(runden_achse, werte, color="#2E7D32", linewidth=2)
    ax.fill_between(runden_achse, werte, alpha=0.2, color="#2E7D32")
    ax.set_xlabel("Trainingsrunde")
    ax.set_ylabel("Wie oft wird ausprobiert (%)")
    ax.set_title("Ausprobieren wird seltener")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


zeige_lernkurve(statistik)

**Was seht ihr im linken Diagramm?** Am Anfang spielt der Agent fast zufällig (um die 33%). Mit der Zeit steigt die Siegquote deutlich über 33% – der Agent hat das Muster des Gegners ausgenutzt!

**Und rechts?** Die "Ausprobieren-Quote" sinkt immer weiter – der Agent vertraut zunehmend dem, was er gelernt hat.

---
## Teil 6: Was genau hat der Agent gelernt?

Schauen wir uns die Punktzahlen direkt an: Was antwortet der Agent, wenn der Gegner zuletzt Schere/Stein/Papier gespielt hat?

In [ ]:
def zeige_gelernte_strategie(agent):
    print("Gelernte Strategie des Agenten\n" + "=" * 50)

    for letzte_gegner_wahl in [SCHERE, STEIN, PAPIER]:
        print(f"\nWenn der Gegner zuletzt {namen[letzte_gegner_wahl]} gespielt hat:")
        punkte = {a: agent.punkte[(letzte_gegner_wahl, a)] for a in [SCHERE, STEIN, PAPIER]}
        beste_antwort = max(punkte, key=punkte.get)

        for antwort in [SCHERE, STEIN, PAPIER]:
            markierung = "  ← Antwort des Agenten" if antwort == beste_antwort else ""
            balken = "█" * max(0, int((punkte[antwort] + 1) * 20))
            print(f"  {namen[antwort]:12s} Punktzahl={punkte[antwort]:+.2f}  {balken}{markierung}")

    # Plot zur besseren Übersicht (ohne Emoji, da Matplotlib diese nicht zuverlässig darstellt)
    namen_plot = {SCHERE: "Schere", STEIN: "Stein", PAPIER: "Papier"}
    fig, ax = plt.subplots(figsize=(8, 4))
    zustaende = [SCHERE, STEIN, PAPIER]
    breite = 0.25
    farben = {"Schere": "#90CAF9", "Stein": "#FFCC80", "Papier": "#A5D6A7"}

    for i, antwort in enumerate([SCHERE, STEIN, PAPIER]):
        werte = [agent.punkte[(z, antwort)] for z in zustaende]
        positionen = [j + (i - 1) * breite for j in range(3)]
        ax.bar(positionen, werte, breite, label=namen_plot[antwort],
               color=list(farben.values())[i], edgecolor="white")

    ax.set_xticks(range(3))
    ax.set_xticklabels([f"Gegner spielte\n{namen_plot[z]}" for z in zustaende])
    ax.set_ylabel("Punktzahl der Antwort")
    ax.set_title("Welche Antwort wählt der Agent in welcher Situation?")
    ax.legend(title="Meine Antwort")
    ax.grid(axis="y", alpha=0.3)
    ax.axhline(y=0, color="gray", linewidth=0.8)
    plt.tight_layout()
    plt.show()


zeige_gelernte_strategie(agent)

**Erkennt ihr die Logik?** Der Gegner wiederholt nach einem Sieg gerne seine Wahl. Der Agent hat gelernt: Wenn der Gegner zuletzt z.B. Stein gespielt hat (und damit eventuell gewonnen hat), kommt mit hoher Wahrscheinlichkeit wieder Stein – also antwortet der Agent am besten mit Papier!

---
## Teil 7: Selbst gegen den Agenten spielen!

In [ ]:
def spiele_gegen_agent(agent, anzahl_runden=10):
    """
    Du spielst gegen den trainierten Agenten.
    Eingabe: s = Schere, t = Stein, p = Papier
    """
    print("=" * 45)
    print("   SCHERE-STEIN-PAPIER – Du vs. KI-Agent")
    print("=" * 45)
    print("Eingabe: s = Schere ✌️   t = Stein ✊   p = Papier ✋\n")

    eingabe_zu_wahl = {"s": SCHERE, "t": STEIN, "p": PAPIER}
    punkte_du = 0
    punkte_agent = 0
    zustand = None  # "was habe ICH zuletzt gespielt" - das beobachtet der Agent ja von mir

    for runde in range(1, anzahl_runden + 1):
        while True:
            eingabe = input(f"Runde {runde}: Deine Wahl (s/t/p): ").strip().lower()
            if eingabe in eingabe_zu_wahl:
                break
            print("Bitte 's', 't' oder 'p' eingeben.")

        deine_wahl = eingabe_zu_wahl[eingabe]
        # Der Agent reagiert auf das, was DU zuletzt gespielt hast
        agent_wahl = max([SCHERE, STEIN, PAPIER], key=lambda a: agent.punkte[(zustand, a)])

        ergebnis = wer_gewinnt(deine_wahl, agent_wahl)
        print(f"  Du: {namen[deine_wahl]}   |   Agent: {namen[agent_wahl]}")

        if ergebnis == 1:
            punkte_du += 1
            print("  ➡️  Du gewinnst die Runde!")
        elif ergebnis == -1:
            punkte_agent += 1
            print("  ➡️  Agent gewinnt die Runde!")
        else:
            print("  ➡️  Unentschieden!")
        print(f"  Stand: Du {punkte_du} – Agent {punkte_agent}\n")

        zustand = deine_wahl

    print("=" * 45)
    if punkte_du > punkte_agent:
        print("🏆 Du hast gewonnen!")
    elif punkte_agent > punkte_du:
        print("🤖 Der Agent hat gewonnen!")
    else:
        print("🤝 Unentschieden!")


# Auskommentieren zum Spielen:
# spiele_gegen_agent(agent)
print("💡 Entferne das '#' in der letzten Zeile und führe die Zelle aus, um gegen den Agenten zu spielen!")
print("💡 Tipp: Versuche, ein eigenes Muster zu verwenden (z.B. nach jedem Sieg wiederholen)")
print("   und schau, ob der Agent es nach ein paar Runden erkennt!")

---
## Teil 8: Selbst ausprobieren

### 🔬 Aufgaben

1. **Anderes Muster:** Ändere in `MusterGegner` die `wiederhol_wahrscheinlichkeit` auf `0.3` (statt `0.7`). Trainiere neu – wird der Agent jetzt schlechter?

2. **Kürzeres Training:** Trainiere nur 2.000 statt 50.000 Runden (`trainiere_agent(anzahl_runden=2000)`). Reicht das, um das Muster zu lernen?

3. **Eigenes Muster spielen:** Spiele in Teil 7 selbst nach einem festen Muster (z.B. immer Schere → Stein → Papier → Schere → ...). Merkt der Agent das?

4. **Anderes Gegnermuster erfinden:** Könnt ihr in `MusterGegner` eine neue Angewohnheit programmieren? Zum Beispiel: "Nach einer Niederlage wechsle ich immer zu der Wahl, die meine letzte geschlagen hätte."

### 💬 Diskussionsfragen

- Warum kann der Agent gegen einen *wirklich* zufälligen Gegner nie mehr als ca. 33% gewinnen?
- Was würde passieren, wenn der Gegner sein Muster mitten im Spiel ändert?
- Wo im echten Leben erkennen Computer ähnliche Muster bei Menschen? *(Tipp: Werbung, Spotify-Empfehlungen, Spiele-KI...)*